# Updated GINO model - 23 June 2026

Targets - $u_x, u_y, u_z$, $\nabla{u}_x, \nabla{u}_y, \nabla{u}_z$


From the predicted particle velocity - use either sr, or a pointwise decoder from latent space for reconstructing flow field. Use the predicted values to propogate the states

Unified GINO training for particle U/gradU and Eulerian reconstruction.

Modes
-----
FINAL2_GINO_TASK=particle_ugradu
    Particle locations are both input geometry and output queries. The model
    predicts [U, gradU] at particles for FLOWUnsteady/rVPM integration.

FINAL2_GINO_TASK=field_reconstruction
    Particle states are mapped to an Eulerian velocity/vorticity field.
    FINAL2_GINO_FIELD_DECODER=super_resolution uses the normal NeuralOperator
    GINO output GNO decoder. FINAL2_GINO_FIELD_DECODER=pointwise uses a
    pointwise query decoder over a learned latent grid.

Channel sweeps
--------------
FINAL2_GINO_INPUT_CHANNELS="Gamma_x,Gamma_y,Gamma_z,sigma,geom_dist,angle_of_attack,phase"
selects exact processed-data feature names. If unset, the defaults below are
used. Coordinates are always supplied through `input_geom`; include x/y/z as
feature channels only if you explicitly want them duplicated.

In [2]:
import socket

print(socket.gethostname())

dysco-navier


In [3]:
import torch.nn.functional as F
from pathlib import Path
import json
import math
import os
import platform
import random
import time
import inspect
from collections import defaultdict
from datetime import datetime

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    from neuralop.models import GINO
    GINO_IMPORT_ERROR = None
except Exception as error:
    GINO = None
    GINO_IMPORT_ERROR = error
    print('[warn] neuralop.models.GINO could not be imported in this kernel.')
    print('       Use the environment/kernel that has NeuralOperator installed.')

try:
    from neuralop.utils import count_model_params
except Exception:
    def count_model_params(model):
        return sum(p.numel() for p in model.parameters())

print('Python      :', platform.python_version())
print('Torch       :', torch.__version__)
print('CUDA build  :', torch.version.cuda)
print('CUDA usable :', torch.cuda.is_available())

Python      : 3.10.12
Torch       : 2.12.1+cu130
CUDA build  : 13.0
CUDA usable : False


Snippet saves the best model in .pt format in the result folder.

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

current_directory = Path.cwd().resolve()
if (current_directory / 'FINAL').is_dir():
    project_folder = current_directory / 'FINAL'
elif current_directory.name == 'FINAL':
    project_folder = current_directory
elif current_directory.parent.name == 'FINAL':
    project_folder = current_directory.parent
else:
    project_folder = current_directory

dataset_candidates = [
    # project_folder / 'processed_data_task1' / 'particle_ugradu_dataset.npz',
    project_folder / 'processed_data'
    # project_folder / 'output' / 'particle_ugradu_dataset.npz',
]
dataset_path = next((candidate for candidate in dataset_candidates if candidate.exists()), dataset_candidates[0])
results_folder = project_folder / 'result'
results_folder.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Project folder:', project_folder)
print('Dataset path  :', dataset_path)
print('Results folder:', results_folder)
print('Device        :', device)
if device.type == 'cuda':
    print('GPU           :', torch.cuda.get_device_name(0))

if not dataset_path.exists():
    raise FileNotFoundError('Missing particle_ugradu_dataset.npz.')

Project folder: /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/FINAL
Dataset path  : /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/FINAL/processed_data
Results folder: /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/FINAL/result
Device        : cpu


I need to define the architecture params here

In [ ]:
CFG = {
    'seed': SEED,
    'file_tag': 'GINO task',
    'epochs': 60,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'eval_every': 5,
    'gradient_accumulation_steps': 4,
    'grad_clip_norm': 1.0,
    'maximum_input_particles': 2000,
    'maximum_train_output_points': 2048,
    'maximum_eval_output_points': 32768,
    'batch_size': 1,
    'num_workers': 0,
    'use_amp': True,
    'latent_res': 16,
    'in_gno_radius': 0.35,
    'out_gno_radius': 0.40,
    'in_gno_transform_type': 'nonlinear_kernelonly',
    'out_gno_transform_type': 'linear',
    'gno_embed_channels': 32,
    'fno_n_modes': (6,6,6),
    'fno_hidden_channels': 32,
    'fno_n_layers': 3,
    'projection_channel_ratio': 2,
    'gno_use_open3d': device.type == 'cuda',
    'gno_use_torch_scatter': device.type == 'cuda',
    # 'gno_use_open3d': False,
    # 'gno_use_torch_scatter': False,
    'debug_dataset_index': 0,
    'debug_batch_every_eval': False,
}

if CFG['batch_size'] != 1:
    print('[warn] GINO supports batching only with shared geometry; forcing batch_size=1 for variable particle clouds.')
    CFG['batch_size'] = 1

CKPT_PATH = results_folder / f"{CFG['file_tag']}_best_model.pt"
LAST_CKPT_PATH = results_folder / f"{CFG['file_tag']}_last_model.pt"
HISTORY_PATH = results_folder / f"{CFG['file_tag']}_history.json"
print('Checkpoint path :', CKPT_PATH)
print('Last-model path :', LAST_CKPT_PATH)
print('History path    :', HISTORY_PATH)
print('Configuration   :')
print(json.dumps(CFG, indent=2))

Checkpoint path : /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/FINAL/result/GINO task_best_model.pt
Last-model path : /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/FINAL/result/GINO task_last_model.pt
History path    : /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/FINAL/result/GINO task_history.json
Configuration   :
{
  "seed": 42,
  "file_tag": "GINO task",
  "epochs": 60,
  "lr": 0.0003,
  "weight_decay": 0.0001,
  "eval_every": 5,
  "gradient_accumulation_steps": 4,
  "grad_clip_norm": 1.0,
  "maximum_input_particles": 2000,
  "maximum_train_output_points": 2048,
  "maximum_eval_output_points": 32768,
  "batch_size": 1,
  "num_workers": 0,
  "use_amp": true,
  "latent_res": 32,
  "in_gno_radius": 0.1,
  "out_gno_radius": 0.12,
  "in_gno_transform_type": "nonlinear_kernelonly",
  "out_gno_transform_type": "linear",
  "gno_embed_channels": 24,
  "fno_n_modes": [
    12,
    12,
    12
  ],
  "fno_hidden_channels": 24,

features and targets from the processed particle frames - raw and normalized arrays

Q: Why should normalization be computed only from the training split of data

Q: Does the geometric encoding really matter? The positions of the particles are a direct channel and they are not predicted. Hence, why will geometry of airfoil matter for this case of prediction of ugradu

Q: The n_particles associated with each frame is the number of particles shed extra during that period, instead shouldn't it be the total number of particles currently in the frame, and why can't that be a feature?


The split is preprocess_data.py is still according to what was written in the Sheets, and has not been changed to radom sampling of train-validate amongst the entire dataset. Fix it in preprocess_data.py

In [ ]:
dataset_file = np.load(dataset_path, allow_pickle=True)

# These are all channels physically stored in the preprocessed .npz file.
# The notebook can select a subset of them for the model without rebuilding the dataset.
raw_feature_names_from_file = [str(name) for name in dataset_file['feature_names'].tolist()]
target_names = [str(name) for name in dataset_file['target_names'].tolist()]
frame_contexts = list(dataset_file['frame_contexts'])
frame_ranges = list(dataset_file['frame_ranges'])

# ------------------------------------------------------------
# Model input-channel controls
# ------------------------------------------------------------
# geom_body_near was found to be weak/non-informative for the current dataset, so it is always removed.
# The remaining geometry channels are kept: geom_dist, geom_nx, geom_ny, geom_nz.
permanently_removed_input_features = [
    'geom_body_near',
]

# Freestream x/z are deterministic functions of AoA when freestream magnitude is fixed.
# Keep this False for the cleaner experiment. Set True only if you want to test whether redundant channels help.
use_freestream_vector_features_as_model_input = False
freestream_vector_feature_names = [
    'freestream_x',
    'freestream_z',
]

removed_by_toggle = [] if use_freestream_vector_features_as_model_input else freestream_vector_feature_names
removed_input_feature_names = sorted(set(permanently_removed_input_features + removed_by_toggle))

# active_input_feature_indices maps the stored .npz columns to the columns actually sent into the model.
active_input_feature_indices = [
    i for i, name in enumerate(raw_feature_names_from_file)
    if name not in removed_input_feature_names
]

# feature_names is intentionally the model-visible feature list, not necessarily every stored column.
feature_names = [raw_feature_names_from_file[i] for i in active_input_feature_indices]
removed_input_features = [
    name for name in raw_feature_names_from_file
    if name not in feature_names
]

print('All stored input features:')
print(raw_feature_names_from_file)
print()
print('Model input features after channel selection:')
print(feature_names)
print()
print('Removed input features:')
print(removed_input_features if removed_input_features else 'none')
print()
print('Freestream vector channels used:', use_freestream_vector_features_as_model_input)
print('Output target names:')
print(target_names)



Q: How should the normalization be performed? Here, it is done framewises.
Why can't it be for one dataset entirely, there is a range of say Gamma, and you normalize the values in each frame according to the global values?

Q: Is this same distribution function correct? Shouldn't it be done in preprocess file?

Validation samples are selected periodically from the temporal sequence of each simulation case

In [ ]:
if 'inputs_by_frame_norm' not in dataset_file or 'targets_by_frame_norm' not in dataset_file:
    raise RuntimeError('This notebook expects frame-wise arrays from preprocess_data_task1.py.')

# Slice input arrays to the active model feature set. Targets are unchanged.
# This removes non-informative/redundant columns selected in the input-channel controls above.
inputs_by_frame_normalized = [
    np.asarray(x, dtype=np.float32)[:, active_input_feature_indices]
    for x in dataset_file['inputs_by_frame_norm'].tolist()
]
targets_by_frame_normalized = [np.asarray(y, dtype=np.float32) for y in dataset_file['targets_by_frame_norm'].tolist()]
inputs_by_frame_raw = [
    np.asarray(x, dtype=np.float32)[:, active_input_feature_indices]
    for x in dataset_file['inputs_by_frame'].tolist()
]
targets_by_frame_raw = [np.asarray(y, dtype=np.float32) for y in dataset_file['targets_by_frame'].tolist()]

training_frame_ids = dataset_file['train_frame_ids'].astype(np.int64)

# Function definition for same distribution sampling for the split.
def make_same_distribution_validation_split(train_ids, stride=5, offset=2, min_per_case=8):
    by_case = {}
    for frame_id in np.asarray(train_ids, dtype=np.int64):
        case = str(frame_ranges[int(frame_id)][0])
        by_case.setdefault(case, []).append(int(frame_id))

    train_out = []
    validation_out = []
    for case, ids in sorted(by_case.items()):
        ids_sorted = sorted(ids, key=lambda j: int(str(frame_ranges[int(j)][1])))
        if len(ids_sorted) <= 1:
            train_out.extend(ids_sorted)
            continue

        stride_value = max(int(stride), 2)
        offset_value = min(max(int(offset), 0), stride_value - 1)
        validation_ids = ids_sorted[offset_value::stride_value]

        minimum_validation = min(max(int(min_per_case), 1), max(len(ids_sorted) - 1, 1))
        if len(validation_ids) < minimum_validation:
            existing = set(validation_ids)
            validation_ids.extend([j for j in ids_sorted if j not in existing][:minimum_validation - len(validation_ids)])

        validation_set = set(validation_ids)
        case_train_ids = [j for j in ids_sorted if j not in validation_set]
        if not case_train_ids:
            case_train_ids = ids_sorted[:-1]
            validation_ids = ids_sorted[-1:]

        train_out.extend(case_train_ids)
        validation_out.extend(validation_ids)

    return np.asarray(sorted(train_out), dtype=np.int64), np.asarray(sorted(validation_out), dtype=np.int64)



In [ ]:
if 'val_id_frame_ids' in dataset_file.files and len(dataset_file['val_id_frame_ids']) > 0:
    validation_frame_ids = dataset_file['val_id_frame_ids'].astype(np.int64)
else:
    print('[warn] val_id_frame_ids is missing/empty in the .npz file.')
    print('[warn] Building same-distribution validation split in-memory from train_frame_ids.')
    print('[warn] For final clean experiments, rerun final-2/preprocess_data_task1.py so this split is saved in the dataset.')
    training_frame_ids, validation_frame_ids = make_same_distribution_validation_split(
        training_frame_ids, stride=5, offset=2, min_per_case=8
    )

validation_angle_frame_ids = dataset_file['validation_angle_frame_ids'].astype(np.int64) if 'validation_angle_frame_ids' in dataset_file.files else dataset_file['val_frame_ids'].astype(np.int64) if 'val_frame_ids' in dataset_file.files else np.array([], dtype=np.int64)
testing_frame_ids = dataset_file['test_frame_ids'].astype(np.int64) if 'test_frame_ids' in dataset_file.files else np.array([], dtype=np.int64)


def frame_context_as_dict(frame_id):
    # np.savez stores dictionaries as object arrays; this helper converts them back safely.
    context = frame_contexts[int(frame_id)]
    if isinstance(context, dict):
        return context
    if hasattr(context, 'item'):
        maybe_dict = context.item()
        if isinstance(maybe_dict, dict):
            return maybe_dict
    return dict(context)



# Reproducible random temporal sampling helper.
# Use this for diagnostics/evaluation subsets so early transient frames do not dominate.
def sample_frame_ids(frame_ids, maximum_frames=None, seed_offset=0, sort_after_sampling=False):
    frame_ids = np.asarray(frame_ids, dtype=np.int64)
    if maximum_frames is None or len(frame_ids) <= int(maximum_frames):
        chosen = frame_ids.copy()
    else:
        rng = np.random.default_rng(SEED + int(seed_offset))
        chosen = rng.choice(frame_ids, size=int(maximum_frames), replace=False)
    if sort_after_sampling:
        chosen = np.asarray(sorted(chosen, key=lambda f: int(float(frame_context_as_dict(int(f)).get('frame', frame_ranges[int(f)][1])))), dtype=np.int64)
    return chosen


def sample_dataset_indices(dataset, maximum_items=None, seed_offset=0):
    indices = np.arange(len(dataset), dtype=np.int64)
    if maximum_items is None or len(indices) <= int(maximum_items):
        return indices
    rng = np.random.default_rng(SEED + int(seed_offset))
    return rng.choice(indices, size=int(maximum_items), replace=False)


def test_frame_ids_for_role(role_name, saved_key, aoa_fallback=None):
    # Prefer explicit preprocessing keys. Fallback keeps older preprocessed files readable.
    if saved_key in dataset_file.files:
        return dataset_file[saved_key].astype(np.int64)
    selected = []
    for frame_id in testing_frame_ids:
        context = frame_context_as_dict(int(frame_id))
        if str(context.get('test_role', '')) == role_name:
            selected.append(int(frame_id))
            continue
        if aoa_fallback is not None and int(round(float(context.get('aoa_deg', -999)))) == int(aoa_fallback):
            selected.append(int(frame_id))
    return np.asarray(selected, dtype=np.int64)

# Testing split - sr, unseen
testing_normal_frame_ids = test_frame_ids_for_role('testing_normal', 'test_normal_frame_ids', aoa_fallback=27)
testing_super_resolution_frame_ids = test_frame_ids_for_role(
    'testing_super_resolution', 'test_super_resolution_frame_ids', aoa_fallback=19
)
testing_unseen_angle_frame_ids = test_frame_ids_for_role('testing_unseen_angle', 'test_unseen_angle_frame_ids', aoa_fallback=32)

# ------------------------------------------------------------
# Remove frames whose complete target vector is zero.
# These are usually startup frame 000000 cases before the particle field has nonzero u/gradU.
# Keeping them makes relative error undefined and gives the optimizer a physically uninformative sample.
# ------------------------------------------------------------
minimum_target_norm_to_keep = 1.0e-12
removed_zero_target_frames = {}


def target_norm_for_frame(frame_id):
    target = targets_by_frame_raw[int(frame_id)]
    return float(np.linalg.norm(target.reshape(-1)))


def filter_zero_target_frames(frame_ids, split_name):
    kept = []
    removed = []
    for frame_id in np.asarray(frame_ids, dtype=np.int64):
        norm_value = target_norm_for_frame(int(frame_id))
        if norm_value > minimum_target_norm_to_keep:
            kept.append(int(frame_id))
        else:
            context = frame_context_as_dict(int(frame_id))
            removed.append({
                'frame_id': int(frame_id),
                'case': str(context.get('case', context.get('case_name', 'unknown'))),
                'frame': str(context.get('frame', context.get('fr', 'unknown'))),
                'target_norm': norm_value,
            })

    removed_zero_target_frames[split_name] = removed
    if removed:
        print(f'[filter] {split_name}: removed {len(removed)} zero-target frames; examples={removed[:3]}')
    else:
        print(f'[filter] {split_name}: no zero-target frames removed')
    return np.asarray(kept, dtype=np.int64)


training_frame_ids = filter_zero_target_frames(training_frame_ids, 'training')
validation_frame_ids = filter_zero_target_frames(validation_frame_ids, 'validation')
validation_angle_frame_ids = filter_zero_target_frames(validation_angle_frame_ids, 'validation_angle')
testing_frame_ids = filter_zero_target_frames(testing_frame_ids, 'testing_all')
testing_normal_frame_ids = filter_zero_target_frames(testing_normal_frame_ids, 'testing_normal')
testing_super_resolution_frame_ids = filter_zero_target_frames(testing_super_resolution_frame_ids, 'testing_super_resolution')
testing_unseen_angle_frame_ids = filter_zero_target_frames(testing_unseen_angle_frame_ids, 'testing_unseen_angle')

has_validation_data = len(validation_frame_ids) > 0
has_testing_data = len(testing_frame_ids) > 0

output_mean = torch.tensor(dataset_file['out_mean'].astype(np.float32), device=device)
output_standard_deviation = torch.tensor(dataset_file['out_std'].astype(np.float32), device=device)

input_dimension = int(inputs_by_frame_normalized[0].shape[1])
output_dimension = int(targets_by_frame_normalized[0].shape[1])



In [ ]:
print('Number of frames      :', len(inputs_by_frame_normalized))
print('Input feature count   :', input_dimension)
print('Output target count   :', output_dimension)
print('Training frames       :', len(training_frame_ids))
print('Validation frames     :', len(validation_frame_ids), '(same-distribution frames from training cases)')
print('Held-out angle validation frames:', len(validation_angle_frame_ids))
print('Testing frames        :', len(testing_frame_ids))
print('  normal test frames  :', len(testing_normal_frame_ids))
print('  super-res test frames:', len(testing_super_resolution_frame_ids))
print('  unseen-angle frames :', len(testing_unseen_angle_frame_ids))
print('Input feature names   :', feature_names)
print('Freestream vector channels used:', use_freestream_vector_features_as_model_input)
print('Removed input features:', removed_input_features if removed_input_features else 'none')
print('Output target names   :', target_names)
if not has_validation_data:
    print('[info] No validation frames yet. The notebook will skip validation plots/metrics until those cases exist.')
if not has_testing_data:
    print('[info] No testing frames yet. The notebook will skip testing plots/metrics until those cases exist.')




In [ ]:
# Silent failures before model training: missing values, wrong shapes, and split leakage.

def check_frame_arrays(frame_ids, split_name, maximum_frames_to_check=50):
    if len(frame_ids) == 0:
        print(f'[{split_name}] no frames available yet; skipping array checks.')
        return

    for frame_id in sample_frame_ids(frame_ids, maximum_frames_to_check, seed_offset=11):
        input_raw = inputs_by_frame_raw[int(frame_id)]
        target_raw = targets_by_frame_raw[int(frame_id)]
        input_normalized = inputs_by_frame_normalized[int(frame_id)]
        target_normalized = targets_by_frame_normalized[int(frame_id)]

        if input_raw.ndim != 2 or input_raw.shape[1] != input_dimension:
            raise RuntimeError(f'{split_name} frame {frame_id}: bad input shape {input_raw.shape}')
        if target_raw.ndim != 2 or target_raw.shape[1] != output_dimension:
            raise RuntimeError(f'{split_name} frame {frame_id}: bad target shape {target_raw.shape}')
        if input_raw.shape[0] != target_raw.shape[0]:
            raise RuntimeError(f'{split_name} frame {frame_id}: input/target particle count mismatch')
        if not np.isfinite(input_raw).all() or not np.isfinite(input_normalized).all():
            raise RuntimeError(f'{split_name} frame {frame_id}: NaN or inf in input')
        if not np.isfinite(target_raw).all() or not np.isfinite(target_normalized).all():
            raise RuntimeError(f'{split_name} frame {frame_id}: NaN or inf in target')

    print(f'[{split_name}] checked {min(len(frame_ids), maximum_frames_to_check)} randomly sampled frames successfully.')


def cases_for_frame_ids(frame_ids):
    cases = []
    for frame_id in np.asarray(frame_ids, dtype=np.int64):
        cases.append(str(frame_ranges[int(frame_id)][0]))
    return set(cases)

check_frame_arrays(training_frame_ids, 'training')
check_frame_arrays(validation_frame_ids, 'validation')
check_frame_arrays(testing_frame_ids, 'testing')

training_cases = cases_for_frame_ids(training_frame_ids)
validation_cases = cases_for_frame_ids(validation_frame_ids)
testing_cases = cases_for_frame_ids(testing_frame_ids)

print('Training cases  :', sorted(training_cases))
print('Validation cases:', sorted(validation_cases))
print('Testing cases   :', sorted(testing_cases))

# Same-distribution validation is intentionally drawn from the training cases.
# It is used for tuning/early stopping, not for final generalization claims.
shared_train_validation_cases = training_cases & validation_cases
print('Shared train/validation cases:', sorted(shared_train_validation_cases))

# Testing must remain case-disjoint from anything used for fitting or tuning.
if training_cases & testing_cases:
    raise RuntimeError('Case leakage: at least one case appears in both training and testing.')
if validation_cases & testing_cases:
    raise RuntimeError('Case leakage: at least one case appears in both validation and testing.')


In [ ]:
# validation/testing target magnitudes live in the same range as training.

def sample_target_magnitudes(frame_ids, maximum_frames=40, maximum_particles_per_frame=30000):
    if len(frame_ids) == 0:
        return None

    chosen_frame_ids = sample_frame_ids(frame_ids, maximum_frames, seed_offset=23)
    velocity_magnitudes = []
    gradient_magnitudes = []

    for frame_id in chosen_frame_ids:
        target = targets_by_frame_raw[int(frame_id)]
        if target.shape[0] > maximum_particles_per_frame:
            picked = np.random.default_rng(SEED + int(frame_id)).choice(
                target.shape[0], size=maximum_particles_per_frame, replace=False
            )
            target = target[picked]

        velocity_magnitudes.append(np.linalg.norm(target[:, :3], axis=1))
        gradient_magnitudes.append(np.linalg.norm(target[:, 3:], axis=1))

    velocity_magnitudes = np.concatenate(velocity_magnitudes)
    gradient_magnitudes = np.concatenate(gradient_magnitudes)
    return {
        'mean_velocity': float(np.mean(velocity_magnitudes)),
        'median_velocity': float(np.median(velocity_magnitudes)),
        'mean_velocity_gradient': float(np.mean(gradient_magnitudes)),
        'median_velocity_gradient': float(np.median(gradient_magnitudes)),
        'velocity_95_percentile': float(np.quantile(velocity_magnitudes, 0.95)),
        'velocity_gradient_95_percentile': float(np.quantile(gradient_magnitudes, 0.95)),
    }

split_statistics = {
    'training': sample_target_magnitudes(training_frame_ids),
    'validation': sample_target_magnitudes(validation_frame_ids),
    'testing': sample_target_magnitudes(testing_frame_ids),
}

print(json.dumps(split_statistics, indent=2))

training_velocity_mean = split_statistics['training']['mean_velocity']
for split_name in ['validation', 'testing']:
    if split_statistics[split_name] is None:
        continue
    ratio = split_statistics[split_name]['mean_velocity'] / max(training_velocity_mean, 1e-12)
    print(f'{split_name} mean |u| / training mean |u| = {ratio:.3f}')

print('')
print('Interpretation: target normalization helps optimization, but it cannot fully remove a physics distribution shift.')
print('If testing magnitudes are very different from training magnitudes, add training cases that cover that range.')


Normalization is with respect to train data alone.

In [ ]:
# Input normalization statistics are stored for all preprocessed columns.
# They may be saved as shape (1, channels), so flatten to a channel vector before slicing.
input_mean_all_channels = dataset_file['in_mean'].astype(np.float32).reshape(-1)
input_standard_deviation_all_channels = dataset_file['in_std'].astype(np.float32).reshape(-1)
input_mean = input_mean_all_channels[active_input_feature_indices]
input_standard_deviation = input_standard_deviation_all_channels[active_input_feature_indices]

# Target statistics are not feature-sliced because all output channels are still predicted.
target_mean = dataset_file['out_mean'].astype(np.float32).reshape(-1)
target_standard_deviation = dataset_file['out_std'].astype(np.float32).reshape(-1)

print('Normalization source: final-2/preprocess_data_task1.py')
print('Relevant lines in preprocessing:')
print('  X_train = concatenate training-frame inputs')
print('  Y_train = concatenate training-frame targets')
print('  in_mean = mean(X_train), in_std = std(X_train)')
print('  out_mean = mean(Y_train), out_std = std(Y_train)')
print('  normalized = (raw - mean) / std')

print('\nFirst input channels:')
for name, mean_value, std_value in zip(feature_names[:min(8, len(feature_names))], input_mean.reshape(-1)[:8], input_standard_deviation.reshape(-1)[:8]):
    print(f'  {name:24s} mean={mean_value: .4e} std={std_value: .4e}')

print('\nOutput channels:')
for name, mean_value, std_value in zip(target_names, target_mean.reshape(-1), target_standard_deviation.reshape(-1)):
    print(f'  {name:24s} mean={mean_value: .4e} std={std_value: .4e}')

# Use a limited number of frames so this audit is fast even for large datasets.
def normalized_channel_summary(frame_ids, arrays_by_frame, maximum_frames=30):
    chosen = sample_frame_ids(frame_ids, maximum_frames, seed_offset=37)
    values = np.concatenate([arrays_by_frame[int(frame_id)] for frame_id in chosen], axis=0)
    return np.mean(values, axis=0), np.std(values, axis=0)

training_input_mean_after_scaling, training_input_std_after_scaling = normalized_channel_summary(
    training_frame_ids, inputs_by_frame_normalized
)
training_target_mean_after_scaling, training_target_std_after_scaling = normalized_channel_summary(
    training_frame_ids, targets_by_frame_normalized
)

print('\nTraining split check after normalization, using a frame sample:')
print('  input mean range :', float(training_input_mean_after_scaling.min()), 'to', float(training_input_mean_after_scaling.max()))
print('  input std range  :', float(training_input_std_after_scaling.min()), 'to', float(training_input_std_after_scaling.max()))
print('  target mean range:', float(training_target_mean_after_scaling.min()), 'to', float(training_target_mean_after_scaling.max()))
print('  target std range :', float(training_target_std_after_scaling.min()), 'to', float(training_target_std_after_scaling.max()))




Plotting the variation of u and gradu for one particle as time progresses for distributions in train and test separately, for one dataset from each.

One can observe near similar plots when AoA is similar. Hence, it is learnable. 

In [ ]:
train_case_name = "26deg_static_airfoil_10u_1p"
test_case_name = "30deg_static_airfoil_10u_1p"

test_frames_source = None

particle_index = 0


def case_name_for_frame(frame_id):
    context = frame_context_as_dict(int(frame_id))
    return str(context.get('case', context.get('case_name', frame_ranges[int(frame_id)][0])))


def numeric_frame_for_sort(frame_id):
    context = frame_context_as_dict(int(frame_id))
    value = context.get('frame', context.get('fr', frame_ranges[int(frame_id)][1]))
    try:
        return int(float(value))
    except Exception:
        digits = ''.join(ch for ch in str(value) if ch.isdigit())
        if digits:
            return int(digits)
        raise ValueError(f'Cannot parse a numeric frame id from frame metadata: {value!r}')


def time_value_for_frame(frame_id):
    context = frame_context_as_dict(int(frame_id))
    dt = float(context.get('dt', 1.0))
    return numeric_frame_for_sort(frame_id) * dt


def available_cases(frame_ids):
    return sorted({case_name_for_frame(int(frame_id)) for frame_id in np.asarray(frame_ids, dtype=np.int64)})


def frames_for_case(frame_ids, case_name):
    selected = [
        int(frame_id)
        for frame_id in np.asarray(frame_ids, dtype=np.int64)
        if case_name_for_frame(int(frame_id)) == str(case_name)
    ]
    return sorted(selected, key=numeric_frame_for_sort)


training_case_options = available_cases(training_frame_ids)
testing_case_options = available_cases(testing_frame_ids)
print('Available training cases:', training_case_options)
print('Available testing cases :', testing_case_options)

if train_case_name is None:
    if not training_case_options:
        raise RuntimeError('No training cases available for temporal target plotting.')
    train_case_name = training_case_options[0]

train_case_frame_ids = frames_for_case(training_frame_ids, train_case_name)
if len(train_case_frame_ids) == 0:
    raise RuntimeError(
        f'No training frames found for train_case_name={train_case_name!r}. '
        f'Available training cases: {training_case_options}'
    )

# The testing source can be either a case-name string or an explicit frame-id array.
if isinstance(test_frames_source, str):
    test_case_name = test_frames_source
    test_candidate_frame_ids = testing_frame_ids
elif test_frames_source is None:
    test_candidate_frame_ids = testing_frame_ids
else:
    test_candidate_frame_ids = np.asarray(test_frames_source, dtype=np.int64)

if test_case_name is None:
    source_cases = available_cases(test_candidate_frame_ids)
    if not source_cases:
        raise RuntimeError('No testing cases available for temporal target plotting.')
    test_case_name = source_cases[0]

test_case_frame_ids = frames_for_case(test_candidate_frame_ids, test_case_name)
if len(test_case_frame_ids) == 0:
    raise RuntimeError(
        f'No testing frames found for test_case_name={test_case_name!r}. '
        f'Available testing cases in chosen source: {available_cases(test_candidate_frame_ids)}'
    )

print(f'Plotting training case: {train_case_name} ({len(train_case_frame_ids)} frames)')
print(f'Plotting testing case : {test_case_name} ({len(test_case_frame_ids)} frames)')


def extract_temporal_series(frame_ids, arrays_by_frame):
    time_values = []
    velocity_values = []
    gradient_values = []

    for frame_id in frame_ids:
        frame_data = arrays_by_frame[int(frame_id)]
        if particle_index >= frame_data.shape[0]:
            continue

        particle = frame_data[particle_index]
        velocity_values.append(np.linalg.norm(particle[:3]))
        gradient_values.append(np.linalg.norm(particle[3:12]))
        time_values.append(time_value_for_frame(frame_id))

    if len(time_values) == 0:
        raise RuntimeError(
            f'particle_index={particle_index} was unavailable in all selected frames. '
            'Try particle_index=0 or another smaller value.'
        )

    return np.asarray(time_values), np.asarray(velocity_values), np.asarray(gradient_values)


train_time, train_velocity, train_gradient = extract_temporal_series(
    train_case_frame_ids,
    targets_by_frame_normalized,
)

test_time, test_velocity, test_gradient = extract_temporal_series(
    test_case_frame_ids,
    targets_by_frame_normalized,
)

figure, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)

axes[0, 0].plot(train_time, train_velocity, linewidth=2)
axes[0, 0].set_title(f'Train: |u| vs time\ncase={train_case_name}, particle={particle_index}')
axes[0, 0].set_xlabel('time')
axes[0, 0].set_ylabel('|u| (normalized)')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(train_time, train_gradient, linewidth=2)
axes[0, 1].set_title(f'Train: |gradU| vs time\ncase={train_case_name}, particle={particle_index}')
axes[0, 1].set_xlabel('time')
axes[0, 1].set_ylabel('|gradU| (normalized)')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(test_time, test_velocity, linewidth=2)
axes[1, 0].set_title(f'Test: |u| vs time\ncase={test_case_name}, particle={particle_index}')
axes[1, 0].set_xlabel('time')
axes[1, 0].set_ylabel('|u| (normalized)')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(test_time, test_gradient, linewidth=2)
axes[1, 1].set_title(f'Test: |gradU| vs time\ncase={test_case_name}, particle={particle_index}')
axes[1, 1].set_xlabel('time')
axes[1, 1].set_ylabel('|gradU| (normalized)')
axes[1, 1].grid(alpha=0.3)

plot_path = results_folder / f'temporal_particle_{particle_index}_target_trace_{train_case_name}_vs_{test_case_name}.png'
figure.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)



Check for abrupt target feature values, small to large norms, zero value, nan

Completely remove the zero time frames which have zero norm

In [ ]:
# Plot target magnitude distributions and check for zero-target frames.

def collect_target_magnitudes_for_plot(frame_ids, maximum_frames=2000, maximum_particles_per_frame=1000000):
    if len(frame_ids) == 0:
        return None
    velocity_values = []
    gradient_values = []
    rng = np.random.default_rng(SEED)

    for frame_id in sample_frame_ids(frame_ids, maximum_frames, seed_offset=53):
        target = targets_by_frame_raw[int(frame_id)]
        if target.shape[0] > maximum_particles_per_frame:
            picked = rng.choice(target.shape[0], size=maximum_particles_per_frame, replace=False)
            target = target[picked]
        velocity_values.append(np.linalg.norm(target[:, :3], axis=1))
        gradient_values.append(np.linalg.norm(target[:, 3:], axis=1))

    return np.concatenate(velocity_values), np.concatenate(gradient_values)


plot_magnitudes = {
    'training': collect_target_magnitudes_for_plot(training_frame_ids),
    'testing': collect_target_magnitudes_for_plot(testing_frame_ids),
}

figure, axes = plt.subplots(1, 2, figsize=(13.5, 4.5), constrained_layout=True)
colors = {'training': '#0278f7', 'validation': '#cdfec5', 'testing': '#e15759'}

for split_name, values in plot_magnitudes.items():
    if values is None:
        continue
    velocity_values, gradient_values = values
    axes[0].hist(velocity_values, bins=80, density=True, histtype='step', linewidth=2.0, color=colors[split_name], label=split_name)
    axes[1].hist(gradient_values, bins=80, density=True, histtype='step', linewidth=2.0, color=colors[split_name], label=split_name)

axes[0].set_title('Velocity magnitude distribution')
axes[0].set_xlabel('|u|')
axes[0].set_ylabel('density')
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].set_title('Gradient of velocity magnitude distribution')
axes[1].set_xlabel('|grad u|')
axes[1].set_ylabel('density')
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

plot_path = results_folder / 'target_magnitude_distributions.png'
figure.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


def target_frame_norms(frame_ids):
    rows = []
    for frame_id in np.asarray(frame_ids, dtype=np.int64):
        target = targets_by_frame_raw[int(frame_id)]
        context = frame_context_as_dict(int(frame_id)) if 'frame_context_as_dict' in globals() else frame_contexts[int(frame_id)]
        rows.append({
            'frame_id': int(frame_id),
            'case': str(context.get('case', 'unknown')),
            'frame': str(context.get('frame', 'unknown')),
            'norm': float(np.linalg.norm(target.reshape(-1))),
            'velocity_norm': float(np.linalg.norm(target[:, :3].reshape(-1))),
            'gradient_norm': float(np.linalg.norm(target[:, 3:].reshape(-1))),
        })
    return rows


for split_name, frame_ids in [('training', training_frame_ids), ('validation', validation_frame_ids), ('testing', testing_frame_ids)]:
    rows = target_frame_norms(sample_frame_ids(frame_ids, 4000, seed_offset=41, sort_after_sampling=True))
    zero_rows = [row for row in rows if row['norm'] <= 1e-12]
    finite_norms = np.asarray([row['norm'] for row in rows], dtype=np.float64)
    if finite_norms.size == 0:
        print(f'{split_name}: no frames')
        continue
    print(
        f"{split_name}: checked first {len(rows)} frames | "
        f"zero-target frames={len(zero_rows)} | "
        f"norm min/median/max={finite_norms.min():.3e}/{np.median(finite_norms):.3e}/{finite_norms.max():.3e}"
    )
    if zero_rows:
        print('  examples:', zero_rows[:3])

Training and Validation nearly overlapping as the number of frames under observation are increased.

Try plotting PDF next

In [ ]:
# Plot target magnitude distributions 2 - train,test,validate.
# If these distributions are far apart, the model is being asked to extrapolate - one case - may or may not fail.

def collect_target_magnitudes_for_plot(frame_ids, maximum_frames=2000, maximum_particles_per_frame=1000000):
    if len(frame_ids) == 0:
        return None
    velocity_values = []
    gradient_values = []
    rng = np.random.default_rng(SEED)

    for frame_id in sample_frame_ids(frame_ids, maximum_frames, seed_offset=53):
        target = targets_by_frame_raw[int(frame_id)]
        if target.shape[0] > maximum_particles_per_frame:
            picked = rng.choice(target.shape[0], size=maximum_particles_per_frame, replace=False)
            target = target[picked]
        velocity_values.append(np.linalg.norm(target[:, :3], axis=1))
        gradient_values.append(np.linalg.norm(target[:, 3:], axis=1))

    return np.concatenate(velocity_values), np.concatenate(gradient_values)

plot_magnitudes = {
    'training': collect_target_magnitudes_for_plot(training_frame_ids),
    # 'validation': collect_target_magnitudes_for_plot(validation_frame_ids),
    'testing': collect_target_magnitudes_for_plot(testing_frame_ids),
}

figure, axes = plt.subplots(1, 2, figsize=(13.5, 4.5), constrained_layout=True)
colors = {'training': "#0278f7", 'validation': "#cdfec5", 'testing': '#e15759'}

for split_name, values in plot_magnitudes.items():
    if values is None:
        continue
    velocity_values, gradient_values = values
    axes[0].hist(velocity_values, bins=80, density=True, histtype='step', linewidth=2.0, color=colors[split_name], label=split_name)
    axes[1].hist(gradient_values, bins=80, density=True, histtype='step', linewidth=2.0, color=colors[split_name], label=split_name)



axes[0].set_title('Velocity magnitude distribution')
axes[0].set_xlabel('|u|')
axes[0].set_ylabel('density')
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].set_title('Gradient of Velocity magnitude distribution')
axes[1].set_xlabel('|gradu|')
axes[1].set_ylabel('density')
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

plot_path = results_folder / 'target_magnitude_distributions.png'
figure.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


In [ ]:
feature_groups = {
    'spatial_coordinates': ['x', 'y', 'z'],

    'vortex_features': [
        'Gamma_x',
        'Gamma_y',
        'Gamma_z',
        'Gamma_mag',
        'sigma',
    ],

    'geometry_features': [
        'geom_dist',
        'geom_body_near',
    ],

    'conditioning_features': [
        'angle_of_attack',
        'freestream_x',
        'freestream_z',
        'phase',
    ],
}


# Keep only existing features
feature_groups = {
    group_name: [
        feature for feature in feature_list
        if feature in feature_names
    ]
    for group_name, feature_list in feature_groups.items()
}


def collect_feature_values(
    frame_ids,
    feature_name,
    normalized=False,
    maximum_frames=5,
    maximum_particles_per_frame=1000000,
):

    if len(frame_ids) == 0:
        return None

    feature_index = feature_names.index(feature_name)

    values = []

    rng = np.random.default_rng(SEED + feature_index)

    source_arrays = (
        inputs_by_frame_normalized
        if normalized
        else inputs_by_frame_raw
    )

    for frame_id in sample_frame_ids(
        frame_ids,
        maximum_frames,
        seed_offset=53,
    ):

        frame = source_arrays[int(frame_id)]

        if frame.shape[0] > maximum_particles_per_frame:

            picked = rng.choice(
                frame.shape[0],
                size=maximum_particles_per_frame,
                replace=False,
            )

            frame = frame[picked]

        values.append(frame[:, feature_index])

    return np.concatenate(values)


# ---------------------------------------------------------
# Publication / ML-paper plotting style
# ---------------------------------------------------------

plt.rcParams.update({
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'legend.fontsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.linewidth': 0.8,
})


splits_for_plot = {
    'training': training_frame_ids,
    'testing': testing_frame_ids,
}


colors = {
    'training': '#1f77b4',
    'testing': '#d62728',
}


line_styles = {
    'training': '-',
    'testing': '--',   # dashed red
}


# ---------------------------------------------------------
# One figure per feature group
# ---------------------------------------------------------

for group_name, feature_list in feature_groups.items():

    if len(feature_list) == 0:
        continue

    n_features = len(feature_list)

    # -----------------------------------------
    # Layout logic
    # -----------------------------------------

    if n_features == 4:
        rows, cols = 2, 2
    else:
        cols = min(3, n_features)
        rows = int(np.ceil(n_features / cols))

    figure, axes = plt.subplots(
        rows,
        cols,
        figsize=(3.8 * cols, 2.8 * rows),
        constrained_layout=True,
    )

    axes = np.asarray(axes).reshape(-1)

    # -----------------------------------------
    # Plot features
    # -----------------------------------------

    for axis, feature_name in zip(axes, feature_list):

        for split_name, frame_ids in splits_for_plot.items():

            values = collect_feature_values(
                frame_ids,
                feature_name,
                normalized=False,
            )

            if values is None:
                continue

            axis.hist(
                values,
                bins=80,
                density=True,
                histtype='step',
                linewidth=1.8,
                linestyle=line_styles[split_name],
                color=colors[split_name],
                alpha=0.95,
                label=split_name,
            )

        # -------------------------------------
        # subplot formatting
        # -------------------------------------

        axis.set_title(feature_name)

        axis.set_ylabel('PDF')

        axis.grid(
            alpha=0.20,
            linestyle=':',
        )

        axis.spines['top'].set_visible(False)
        axis.spines['right'].set_visible(False)

        # Put x-axis labels BELOW the axis
        axis.tick_params(
            axis='x',
            direction='out',
            pad=5,
            bottom=True,
            top=False,
            labelbottom=True,
        )

        axis.xaxis.set_ticks_position('bottom')

        axis.legend(
            frameon=False,
            loc='best',
        )

    # -----------------------------------------
    # Remove empty axes
    # -----------------------------------------

    for axis in axes[n_features:]:
        axis.axis('off')

    # -----------------------------------------
    # Figure title
    # -----------------------------------------

    figure.suptitle(
        group_name.replace('_', ' ').title(),
        fontsize=12,
    )

    plot_path = (
        results_folder
        / f'{group_name}_distribution_plot.png'
    )

    figure.savefig(
        plot_path,
        dpi=400,
        bbox_inches='tight',
    )

    plt.show()

    print('Saved:', plot_path)

Data-pair visualization for GNO learning

In [ ]:
from matplotlib.colors import LogNorm

plt.rcParams.update({
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'axes.linewidth': 0.8,
    'figure.titlesize': 12,
})


split_frame_ids = {
    'training': training_frame_ids,
    # 'validation': validation_frame_ids,
    # 'validation_angle': validation_angle_frame_ids,
    # 'testing_normal': testing_normal_frame_ids,
    # 'testing_super_resolution': testing_super_resolution_frame_ids,
    # 'testing_unseen_angle': testing_unseen_angle_frame_ids,
}


split_colors = {
    'training': '#4c78a8',
    'validation': '#59a14f',
    'validation_angle': '#edc948',
    'testing_normal': '#e15759',
    'testing_super_resolution': '#b07aa1',
    'testing_unseen_angle': '#f28e2b',
}


# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------

def feature_index_or_none(name):

    return (
        feature_names.index(name)
        if name in feature_names
        else None
    )


def target_index_or_none(name):

    return (
        target_names.index(name)
        if name in target_names
        else None
    )


def context_for_frame(frame_id):

    context = frame_contexts[int(frame_id)]

    return (
        context
        if isinstance(context, dict)
        else dict(context.item())
    )


def choose_representative_frame(frame_ids, split_name):

    if len(frame_ids) == 0:
        return None

    ordered = np.asarray(frame_ids, dtype=np.int64)

    # avoid earliest transient frames
    return int(ordered[len(ordered) // 2])


def sample_points_for_plot(
    input_raw,
    target_raw,
    maximum_points=9000,
    seed_offset=0,
):

    particle_count = input_raw.shape[0]

    if particle_count <= maximum_points:
        return input_raw, target_raw

    rng = np.random.default_rng(SEED + seed_offset)

    chosen = rng.choice(
        particle_count,
        size=maximum_points,
        replace=False,
    )

    return input_raw[chosen], target_raw[chosen]


def robust_color_limits(values, lower=2.0, upper=98.0):

    values = np.asarray(values)

    finite = values[np.isfinite(values)]

    if finite.size == 0:
        return 0.0, 1.0

    low, high = np.percentile(
        finite,
        [lower, upper],
    )

    if np.isclose(low, high):
        low, high = (
            float(finite.min()),
            float(finite.max()),
        )

    if np.isclose(low, high):
        high = low + 1.0

    return float(low), float(high)

def add_3d_cloud(
    axis,
    coordinates,
    color_values,
    title,
    colorbar_label,
    cmap='viridis',
    use_log=False,
):

    low, high = robust_color_limits(color_values)

    scatter_kwargs = dict(
        c=color_values,
        s=1.2,
        alpha=0.58,
        cmap=cmap,
        linewidths=0.0,
    )

    if use_log:

        scatter_kwargs['norm'] = LogNorm(
            vmin=max(low, 1e-6),
            vmax=high,
        )

    else:

        scatter_kwargs['vmin'] = low
        scatter_kwargs['vmax'] = high

    plot = axis.scatter(
        coordinates[:, 0],
        coordinates[:, 1],
        coordinates[:, 2],
        **scatter_kwargs,
    )


    axis.set_title(title, pad=6)


    axis.set_xlabel('x', labelpad=-4)
    axis.set_ylabel('y', labelpad=-4)
    axis.set_zlabel('z', labelpad=-4)

    axis.view_init(
        elev=20,
        azim=120,
    )


    axis.set_box_aspect([1, 1, 1])


    axis.xaxis.pane.fill = False
    axis.yaxis.pane.fill = False
    axis.zaxis.pane.fill = False

    axis.grid(False)


    axis.tick_params(
        axis='x',
        direction='out',
        pad=1,
        bottom=True,
        top=False,
        labelbottom=True,
    )

    colorbar = plt.colorbar(
        plot,
        ax=axis,
        shrink=0.72,
        pad=0.01,
        fraction=0.045,
    )

    colorbar.set_label(
        colorbar_label,
        fontsize=8,
    )

    colorbar.ax.tick_params(labelsize=7)


def plot_input_output_pair(split_name, frame_ids):

    frame_id = choose_representative_frame(
        frame_ids,
        split_name,
    )

    if frame_id is None:

        print(f'[{split_name}] no frames available.')

        return

    input_raw = inputs_by_frame_raw[frame_id]

    target_raw = targets_by_frame_raw[frame_id]

    input_plot, target_plot = sample_points_for_plot(
        input_raw,
        target_raw,
        seed_offset=frame_id,
    )

    x_i = feature_index_or_none('x')
    y_i = feature_index_or_none('y')
    z_i = feature_index_or_none('z')

    if None in (x_i, y_i, z_i):

        raise RuntimeError(
            'Expected x/y/z input features.'
        )

    coordinates = input_plot[:, [x_i, y_i, z_i]]

    # -----------------------------------------------------
    # circulation magnitude
    # -----------------------------------------------------

    gamma_columns = [
        feature_index_or_none(name)
        for name in [
            'Gamma_x',
            'Gamma_y',
            'Gamma_z',
        ]
    ]

    if all(index is not None for index in gamma_columns):

        circulation_color = np.linalg.norm(
            input_plot[:, gamma_columns],
            axis=1,
        )

        circulation_label = '|Gamma|'

    elif feature_index_or_none('Gamma_mag') is not None:

        circulation_color = input_plot[
            :,
            feature_index_or_none('Gamma_mag')
        ]

        circulation_label = 'Gamma_mag'

    else:

        circulation_color = np.zeros(
            input_plot.shape[0],
            dtype=np.float32,
        )

        circulation_label = 'circulation unavailable'

    # -----------------------------------------------------
    # sigma
    # -----------------------------------------------------

    sigma_i = feature_index_or_none('sigma')

    sigma_color = (
        input_plot[:, sigma_i]
        if sigma_i is not None
        else np.zeros(
            input_plot.shape[0],
            dtype=np.float32,
        )
    )

    # -----------------------------------------------------
    # targets
    # -----------------------------------------------------

    velocity_magnitude = np.linalg.norm(
        target_plot[:, :3],
        axis=1,
    )

    gradient_magnitude = np.linalg.norm(
        target_plot[:, 3:],
        axis=1,
    )

    # -----------------------------------------------------
    # metadata
    # -----------------------------------------------------

    context = context_for_frame(frame_id)

    case_name = str(
        context.get(
            'case',
            'unknown case',
        )
    )

    physical_frame = str(
        context.get(
            'frame',
            context.get('fr', 'unknown frame'),
        )
    )

    particle_count = int(
        context.get(
            'n_particles',
            input_raw.shape[0],
        )
    )

    figure = plt.figure(
        figsize=(11, 8),
        constrained_layout=True,
    )

    axes = [
        figure.add_subplot(
            2,
            2,
            i + 1,
            projection='3d',
        )
        for i in range(4)
    ]


    add_3d_cloud(
        axes[0],
        coordinates,
        circulation_color,
        'Circulation magnitude',
        circulation_label,
        cmap='coolwarm',
    )

    add_3d_cloud(
        axes[1],
        coordinates,
        sigma_color,
        'Core radius',
        'sigma',
        cmap='viridis',
    )

    add_3d_cloud(
        axes[2],
        coordinates,
        velocity_magnitude,
        'Velocity magnitude',
        '|u|',
        cmap='viridis',
    )

    add_3d_cloud(
        axes[3],
        coordinates,
        gradient_magnitude,
        'Velocity-gradient magnitude',
        '|gradU|',
        cmap='magma',
        use_log=True,
    )


    figure.suptitle(
        (
            f'{split_name} | '
            f'{case_name} | '
            f'frame {physical_frame} | '
            f'N={particle_count:,}'
        ),
        fontsize=12,
    )

    plot_path = (
        results_folder
        / f'{split_name.lower()}_publication_cloud.png'
    )

    figure.savefig(
        plot_path,
        dpi=350,
        bbox_inches='tight',
    )

    plt.show()

    print('Saved:', plot_path)


for split_name, frame_ids in split_frame_ids.items():

    plot_input_output_pair(
        split_name,
        frame_ids,
    )

50% of frames have 95th percentile |u| well below 3 - implied from cdf value plot

In [ ]:
# Split-level distribution diagnostics for GNO learning.
# These plots reveal whether validation/testing require interpolation or extrapolation in particle count,
# AoA/phase, local geometry, and target magnitude.


def collect_frame_level_summary(frame_ids, maximum_frames=2000, maximum_particles_per_frame=1000000):
    rows = []
    rng = np.random.default_rng(SEED + 991)
    for frame_id in sample_frame_ids(frame_ids, maximum_frames, seed_offset=53):
        input_raw = inputs_by_frame_raw[int(frame_id)]
        target_raw = targets_by_frame_raw[int(frame_id)]
        if input_raw.shape[0] > maximum_particles_per_frame:
            chosen = rng.choice(input_raw.shape[0], size=maximum_particles_per_frame, replace=False)
            input_used = input_raw[chosen]
            target_used = target_raw[chosen]
        else:
            input_used = input_raw
            target_used = target_raw

        context = context_for_frame(int(frame_id))
        row = {
            'frame_id': int(frame_id),
            'case': str(context.get('case', 'unknown')),
            'physical_frame': float(context.get('frame', context.get('fr', np.nan))),
            'particle_count': int(input_raw.shape[0]),
            'velocity_median': float(np.median(np.linalg.norm(target_used[:, :3], axis=1))),
            'velocity_95': float(np.percentile(np.linalg.norm(target_used[:, :3], axis=1), 95)),
            'gradient_median': float(np.median(np.linalg.norm(target_used[:, 3:], axis=1))),
            'gradient_95': float(np.percentile(np.linalg.norm(target_used[:, 3:], axis=1), 95)),
        }

        for optional_feature in ['angle_of_attack', 'phase', 'geom_dist', 'geom_body_near', 'nearest_particle_distance', 'local_neighbor_count']:
            feature_i = feature_index_or_none(optional_feature)
            if feature_i is not None:
                row[optional_feature] = float(np.median(input_used[:, feature_i]))
        rows.append(row)
    return rows


summary_by_split = {
    split_name: collect_frame_level_summary(frame_ids)
    for split_name, frame_ids in split_frame_ids.items()
}

print('Frame-level summary quantiles')
for split_name, rows in summary_by_split.items():
    if len(rows) == 0:
        print(f'  {split_name}: no frames')
        continue
    particle_counts = np.asarray([row['particle_count'] for row in rows], dtype=np.float64)
    velocity_95 = np.asarray([row['velocity_95'] for row in rows], dtype=np.float64)
    gradient_95 = np.asarray([row['gradient_95'] for row in rows], dtype=np.float64)
    print(
        f"  {split_name}: frames={len(rows)}, "
        f"particles median={np.median(particle_counts):.0f}, "
        f"|u|95 median={np.median(velocity_95):.4g}, "
        f"|gradU|95 median={np.median(gradient_95):.4g}"
    )

figure, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

for split_name, rows in summary_by_split.items():
    if len(rows) == 0:
        continue
    color = split_colors[split_name]
    particle_counts = np.asarray([row['particle_count'] for row in rows], dtype=np.float64)
    velocity_95 = np.asarray([row['velocity_95'] for row in rows], dtype=np.float64)
    gradient_95 = np.asarray([row['gradient_95'] for row in rows], dtype=np.float64)
    frames = np.asarray([row['physical_frame'] for row in rows], dtype=np.float64)

    axes[0, 0].hist(particle_counts, bins=35, histtype='step', linewidth=2.0, color=color, label=split_name)
    axes[0, 1].scatter(frames, velocity_95, s=18, alpha=0.75, color=color, label=split_name)
    axes[1, 0].scatter(frames, gradient_95, s=18, alpha=0.75, color=color, label=split_name)

    sorted_velocity = np.sort(velocity_95)
    cdf = np.linspace(0.0, 1.0, sorted_velocity.size, endpoint=True)
    axes[1, 1].plot(sorted_velocity, cdf, linewidth=2.0, color=color, label=split_name)

axes[0, 0].set_title('Particle count per frame')
axes[0, 0].set_xlabel('particles')
axes[0, 0].set_ylabel('frame count')
axes[0, 0].grid(alpha=0.25)
axes[0, 0].legend(frameon=False)

axes[0, 1].set_title('Velocity target scale over time')
axes[0, 1].set_xlabel('simulation frame')
axes[0, 1].set_ylabel('95th percentile |u|')
axes[0, 1].grid(alpha=0.25)
axes[0, 1].legend(frameon=False)

axes[1, 0].set_title('Velocity-gradient target scale over time')
axes[1, 0].set_xlabel('simulation frame')
axes[1, 0].set_ylabel('95th percentile |gradU|')
axes[1, 0].grid(alpha=0.25)
axes[1, 0].legend(frameon=False)

axes[1, 1].set_title('CDF of high-end velocity target scale')
axes[1, 1].set_xlabel('95th percentile |u| per frame')
axes[1, 1].set_ylabel('cumulative probability')
axes[1, 1].grid(alpha=0.25)
axes[1, 1].legend(frameon=False)

plot_path = results_folder / 'gno_data_pair_distribution_summary.png'
figure.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)

# Optional conditioning/geometry panel: drawn only for features present in the preprocessed file.
optional_panels = [name for name in ['angle_of_attack', 'phase', 'geom_dist', 'nearest_particle_distance', 'local_neighbor_count'] if name in feature_names]
if optional_panels:
    columns = min(3, len(optional_panels))
    rows = int(np.ceil(len(optional_panels) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(5.2 * columns, 3.5 * rows), constrained_layout=True)
    axes = np.asarray(axes).reshape(-1)

    for axis, feature_name in zip(axes, optional_panels):
        for split_name, frame_ids in split_frame_ids.items():
            values = collect_feature_values(frame_ids, feature_name, normalized=False, maximum_frames=2000, maximum_particles_per_frame=1000000)
            if values is None:
                continue
            axis.hist(values, bins=60, density=True, histtype='step', linewidth=1.9, color=split_colors[split_name], label=split_name)
        axis.set_title(feature_name)
        axis.set_ylabel('density')
        axis.grid(alpha=0.25)

    for axis in axes[len(optional_panels):]:
        axis.axis('off')
    axes[0].legend(frameon=False)
    figure.suptitle('Conditioning and geometry distributions used by the GNO', fontsize=14)
    plot_path = results_folder / 'gno_conditioning_geometry_distributions.png'
    figure.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)
else:
    print('No conditioning/geometry features found for the optional distribution panel.')


In [ ]:
# Compute coordinate normalization from training frames
all_train_xyz = []
for fid in training_frame_ids:
    raw = inputs_by_frame_raw[int(fid)]
    all_train_xyz.append(raw[:, :3])
all_train_xyz = np.vstack(all_train_xyz)
coord_min = np.min(all_train_xyz, axis=0).astype(np.float32)
coord_max = np.max(all_train_xyz, axis=0).astype(np.float32)
coord_span = np.maximum(coord_max - coord_min, 1e-8)

def normalize_xyz(xyz):
    return (xyz.astype(np.float32) - coord_min) / coord_span